In [1]:
import pandas as pd

print("Manufacturing Data Quality Monitor")
print("Pandas version:", pd.__version__)
print("Ready to build!")

Manufacturing Data Quality Monitor
Pandas version: 2.2.2
Ready to build!


In [2]:

from pathlib import Path
import pandas as pd

# 1. Create a folder for our project data
output_folder = Path("data")
output_folder.mkdir(exist_ok=True)

# 2. Generate 20 fictional batch start times
start_times = pd.date_range(
    start="2026-10-01 06:00:00",
    periods=20,
    freq="45min"
)

# 3. Build the manufacturing records
data = []

for i in range(20):
    start = start_times[i]

    data.append({
        "batch_id": f"BATCH-{1001 + i}",
        "production_line": f"LINE-{(i % 3) + 1}",
        "product": ["Product A", "Product B", "Product C"][i % 3],
        "start_time": start,
        "end_time": start + pd.Timedelta(
            minutes=30 + (i % 5) * 10
        ),
        "quantity": 100 + i * 10,
        "status": "Completed"
    })

df = pd.DataFrame(data)

# 4. Introduce simulated data-quality problems
df.loc[2, "start_time"] = pd.NaT
df.loc[5, "end_time"] = pd.NaT
df.loc[8, "end_time"] = (
    df.loc[8, "start_time"] - pd.Timedelta(minutes=15)
)
df.loc[11, "quantity"] = -50

# Add one duplicate record
df = pd.concat([df, df.iloc[[4]]], ignore_index=True)

# 5. Save the raw data as a CSV file
file_path = output_folder / "mes_batch_records_raw.csv"
df.to_csv(file_path, index=False)

# 6. Display the results
print("Dataset created successfully!")
print("Total records:", len(df))
print("Total columns:", len(df.columns))
print("Saved to:", file_path)

display(df.head(10))


Dataset created successfully!
Total records: 21
Total columns: 7
Saved to: data\mes_batch_records_raw.csv


,batch_id,production_line,product,start_time,end_time,quantity,status
0,BATCH-1001,LINE-1,Product A,2026-10-01 06:00:00,2026-10-01 06:30:00,100,Completed
1,BATCH-1002,LINE-2,Product B,2026-10-01 06:45:00,2026-10-01 07:25:00,110,Completed
2,BATCH-1003,LINE-3,Product C,NaT,2026-10-01 08:20:00,120,Completed
3,BATCH-1004,LINE-1,Product A,2026-10-01 08:15:00,2026-10-01 09:15:00,130,Completed
4,BATCH-1005,LINE-2,Product B,2026-10-01 09:00:00,2026-10-01 10:10:00,140,Completed
5,BATCH-1006,LINE-3,Product C,2026-10-01 09:45:00,NaT,150,Completed
6,BATCH-1007,LINE-1,Product A,2026-10-01 10:30:00,2026-10-01 11:10:00,160,Completed
7,BATCH-1008,LINE-2,Product B,2026-10-01 11:15:00,2026-10-01 12:05:00,170,Completed
8,BATCH-1009,LINE-3,Product C,2026-10-01 12:00:00,2026-10-01 11:45:00,180,Completed
9,BATCH-1010,LINE-1,Product A,2026-10-01 12:45:00,2026-10-01 13:55:00,190,Completed


In [3]:

# 1. Create a working copy of our raw dataset
df_check = df.copy()

# 2. Check for missing values
print("MISSING VALUES PER COLUMN")
print(df_check.isna().sum())

# 3. Count duplicate records
duplicate_count = df_check.duplicated().sum()
print("\nDUPLICATE ROWS:", duplicate_count)

# 4. Identify invalid batch timestamps
invalid_timestamps = (
    df_check["start_time"].notna()
    & df_check["end_time"].notna()
    & (df_check["end_time"] < df_check["start_time"])
)

print("INVALID TIMESTAMP ROWS:", invalid_timestamps.sum())

# 5. Identify negative production quantities
negative_quantity = df_check["quantity"] < 0
print("NEGATIVE QUANTITY ROWS:", negative_quantity.sum())

# 6. Create a summary of the quality checks
quality_report = pd.DataFrame({
    "check": [
        "Missing start time",
        "Missing end time",
        "Duplicate records",
        "End time before start time",
        "Negative quantity"
    ],
    "issue_count": [
        df_check["start_time"].isna().sum(),
        df_check["end_time"].isna().sum(),
        duplicate_count,
        invalid_timestamps.sum(),
        negative_quantity.sum()
    ]
})

print("\nDATA QUALITY REPORT")
display(quality_report)


MISSING VALUES PER COLUMN
batch_id           0
production_line    0
product            0
start_time         1
end_time           1
quantity           0
status             0
dtype: int64

DUPLICATE ROWS: 1
INVALID TIMESTAMP ROWS: 1
NEGATIVE QUANTITY ROWS: 1

DATA QUALITY REPORT


,check,issue_count
0,Missing start time,1
1,Missing end time,1
2,Duplicate records,1
3,End time before start time,1
4,Negative quantity,1


In [4]:

# 1. Work on a copy; preserve our original raw dataset
df_analysis = df_check.copy()

# 2. Identify batches with valid timestamps
valid_duration = (
    df_analysis["start_time"].notna()
    & df_analysis["end_time"].notna()
    & (df_analysis["end_time"] >= df_analysis["start_time"])
)

# 3. Calculate duration in minutes
df_analysis["duration_minutes"] = (
    df_analysis["end_time"] - df_analysis["start_time"]
).dt.total_seconds() / 60

# Do not report durations for invalid batches
df_analysis.loc[~valid_duration, "duration_minutes"] = pd.NA

print("BATCH DURATION RESULTS")
display(
    df_analysis[
        ["batch_id", "production_line", "duration_minutes"]
    ]
)

# 4. Prepare production data
# Our simulated duplicate is an exact copy, so retain one per batch ID
df_unique = df_analysis.drop_duplicates(subset=["batch_id"]).copy()

# Exclude invalid quantities from production totals
valid_quantity = df_unique["quantity"] >= 0
df_production = df_unique.loc[valid_quantity].copy()

# 5. Summarize production by line
production_summary = (
    df_production.groupby("production_line")
    .agg(
        batch_count=("batch_id", "nunique"),
        total_quantity=("quantity", "sum"),
        average_batch_duration=("duration_minutes", "mean")
    )
    .reset_index()
)

production_summary["average_batch_duration"] = (
    production_summary["average_batch_duration"].round(2)
)

print("\nPRODUCTION SUMMARY BY LINE")
display(production_summary)


BATCH DURATION RESULTS


,batch_id,production_line,duration_minutes
0,BATCH-1001,LINE-1,30.0
1,BATCH-1002,LINE-2,40.0
2,BATCH-1003,LINE-3,NaN
3,BATCH-1004,LINE-1,60.0
4,BATCH-1005,LINE-2,70.0
5,BATCH-1006,LINE-3,NaN
6,BATCH-1007,LINE-1,40.0
7,BATCH-1008,LINE-2,50.0
8,BATCH-1009,LINE-3,NaN
9,BATCH-1010,LINE-1,70.0



PRODUCTION SUMMARY BY LINE


,production_line,batch_count,total_quantity,average_batch_duration
0,LINE-1,7,1330,48.57
1,LINE-2,7,1400,51.43
2,LINE-3,5,960,60.00


In [5]:

# Export the project results
quality_report.to_csv(
    output_folder / "data_quality_report.csv",
    index=False
)

production_summary.to_csv(
    output_folder / "production_summary_by_line.csv",
    index=False
)

df_analysis[
    [
        "batch_id",
        "production_line",
        "product",
        "start_time",
        "end_time",
        "duration_minutes",
        "quantity",
        "status"
    ]
].to_csv(
    output_folder / "batch_duration_analysis.csv",
    index=False
)

print("PROJECT OUTPUTS EXPORTED")
print("\nFiles created:")

for file in sorted(output_folder.glob("*.csv")):
    print(f"- {file.name}")

print("\nProduction summary exported successfully!")


PROJECT OUTPUTS EXPORTED

Files created:
- batch_duration_analysis.csv
- data_quality_report.csv
- mes_batch_records_raw.csv
- production_summary_by_line.csv
- record_level_validation_report.csv

Production summary exported successfully!


In [6]:

# Build a report showing exactly which records have problems

issues = []

# Identify rows that are part of an exact duplicate group
duplicate_rows = df_check.duplicated(keep=False)

for idx, row in df_check.iterrows():

    # Missing start timestamp
    if pd.isna(row["start_time"]):
        issues.append({
            "row_index": idx,
            "batch_id": row["batch_id"],
            "issue": "Missing start time"
        })

    # Missing end timestamp
    if pd.isna(row["end_time"]):
        issues.append({
            "row_index": idx,
            "batch_id": row["batch_id"],
            "issue": "Missing end time"
        })

    # End timestamp earlier than start timestamp
    if (
        pd.notna(row["start_time"])
        and pd.notna(row["end_time"])
        and row["end_time"] < row["start_time"]
    ):
        issues.append({
            "row_index": idx,
            "batch_id": row["batch_id"],
            "issue": "End time before start time"
        })

    # Negative production quantity
    if pd.notna(row["quantity"]) and row["quantity"] < 0:
        issues.append({
            "row_index": idx,
            "batch_id": row["batch_id"],
            "issue": "Negative quantity"
        })

    # Exact duplicate record
    if duplicate_rows.loc[idx]:
        issues.append({
            "row_index": idx,
            "batch_id": row["batch_id"],
            "issue": "Duplicate record"
        })

# Convert the issues into a DataFrame
validation_report = pd.DataFrame(
    issues,
    columns=["row_index", "batch_id", "issue"]
)

# Export the detailed validation report
validation_report.to_csv(
    output_folder / "record_level_validation_report.csv",
    index=False
)

print("RECORD-LEVEL VALIDATION REPORT")
print("Total issues identified:", len(validation_report))
display(validation_report)

print("\nIssue counts:")
display(
    validation_report["issue"].value_counts().rename_axis("issue")
    .reset_index(name="count")
)


RECORD-LEVEL VALIDATION REPORT
Total issues identified: 6


,row_index,batch_id,issue
0,2,BATCH-1003,Missing start time
1,4,BATCH-1005,Duplicate record
2,5,BATCH-1006,Missing end time
3,8,BATCH-1009,End time before start time
4,11,BATCH-1012,Negative quantity
5,20,BATCH-1005,Duplicate record



Issue counts:


,issue,count
0,Duplicate record,2
1,Missing start time,1
2,Missing end time,1
3,End time before start time,1
4,Negative quantity,1


In [7]:
print("Kernel is working!")

Kernel is working!
